In [2]:
import os
import pandas as pd
import pydicom
import SimpleITK as sitk
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
import subprocess
from pathlib import Path
from tqdm import tqdm

# PROSTATEx: Convert DICOM to NIfTI
This notebook converts PROSTATEx DICOM files into NIfTI format, organizing them by patient study and MRI modality (T2, ADC, high b-value, individual b-value diffusion series).

In [3]:
dicom_root = '/workspace/data/dicom/'
nifti_root = '/workspace/data/nifti/'
os.makedirs(nifti_root, exist_ok=True)

In [ ]:
pats = os.listdir(dicom_root)
print(f'Found {len(pats)} patients in DICOM directory.')

In [ ]:
# Build a mapping of (patient_studyDate) -> {modality: series_folder_name}
# by scanning DICOM directory structure and classifying series by folder name keywords
dicom_series = dict()
for pat in pats:
    folders = os.listdir(os.path.join(dicom_root, pat))
    for folder in folders:
        folder_date = folder[:10]
        dicom_series[pat+"_"+folder_date] = dict()
        series = os.listdir(os.path.join(dicom_root, pat, folder))
        for s in series:
            # Classify diffusion-related series (DYN, diff, 4bval, DIFF)
            if 'DYN' in s or 'diff' in s or '4bval' in s or 'DIFF' in s:
                if 'ADC' in s:
                    dicom_series[pat+"_"+folder_date]['ADC'] = s
                elif 'BVAL' in s:
                    dicom_series[pat+"_"+folder_date]['HBV'] = s
                else:
                    # Raw diffusion series containing multiple b-values
                    dicom_series[pat+"_"+folder_date]['Bs'] = s
            # Classify T2-weighted series by orientation
            if 't2' in s:
                if 'tra' in s:
                    dicom_series[pat+"_"+folder_date]['T2'] = s
                elif 'sag' in s:
                    dicom_series[pat+"_"+folder_date]['T2Sag'] = s
                elif 'cor' in s:
                    dicom_series[pat+"_"+folder_date]['T2Cor'] = s

In [ ]:
# Convert each DICOM series to NIfTI format
for study, series in tqdm(dicom_series.items()):
    study_folder = Path(nifti_root) / study
    study_folder.mkdir(parents=True, exist_ok=True)

    patient = study.split("_")[0]
    study_date = study.split("_")[1]
    patient_dicom_root = Path(dicom_root) / patient
    
    for folder in patient_dicom_root.iterdir():
        if folder.name[:10] != study_date:
            continue

        for modality, series_name in series.items():
            dicom_series_folder = folder / series_name

            if modality == 'Bs':
                # Group DICOM files by b-value extracted from the Sequence Name tag (0018,0024)
                b_value_groups = defaultdict(list)
                dicom_files = list(dicom_series_folder.glob('*.dcm'))

                for dicom_file_path in dicom_files:
                    dicom_file = pydicom.dcmread(dicom_file_path)
                    b_value_str = dicom_file[(0x0018, 0x0024)].value
                    b_value = 'B' + b_value_str.split('_')[1][1:][:-1]
                    b_value_groups[b_value].append(str(dicom_file_path))

                # Convert each b-value group to a separate NIfTI file
                for b_value, file_paths in b_value_groups.items():
                    reader = sitk.ImageSeriesReader()
                    reader.SetFileNames(file_paths[::-1])
                    image = reader.Execute()

                    nifti_filename = study_folder / f'{b_value}.nii.gz'
                    sitk.WriteImage(image, str(nifti_filename))

            else:
                # Standard conversion for non-diffusion modalities (T2, ADC, HBV, etc.)
                reader = sitk.ImageSeriesReader()
                dicom_filenames = reader.GetGDCMSeriesFileNames(str(dicom_series_folder))
                reader.SetFileNames(dicom_filenames)
                image = reader.Execute()

                nifti_filename = study_folder / f'{modality}.nii.gz'
                sitk.WriteImage(image, str(nifti_filename))

print("Conversion completed.")

## Re-convert problematic studies
Some studies have incorrect slice ordering in the diffusion series. Re-run conversion for these specific studies without reversing file paths.

In [ ]:
# Studies whose diffusion slices need non-reversed ordering
reconv_studies = [
    'ProstateX-0200_08-12-2011',
    'ProstateX-0201_09-01-2011',
    'ProstateX-0202_09-01-2011',
    'ProstateX-0203_09-05-2011',
    'ProstateX-0218_02-18-2011',
    'ProstateX-0219_03-03-2011',
    'ProstateX-0220_03-03-2011',
    'ProstateX-0221_03-09-2011',
    'ProstateX-0224_03-09-2011',
    'ProstateX-0225_03-21-2011',
    'ProstateX-0226_04-03-2011',
    'ProstateX-0227_04-27-2011',
    'ProstateX-0228_04-29-2011',
    'ProstateX-0229_04-29-2011',
    'ProstateX-0230_05-03-2011',
    'ProstateX-0231_05-03-2011',
    'ProstateX-0232_05-12-2011',
    'ProstateX-0233_05-12-2011',
    'ProstateX-0235_05-14-2011',
    'ProstateX-0236_05-19-2011',
    'ProstateX-0238_05-24-2011',
    'ProstateX-0237_05-21-2011',
    'ProstateX-0239_05-25-2011',
    'ProstateX-0240_05-26-2011',
    'ProstateX-0241_05-29-2011',
    'ProstateX-0242_06-04-2011',
    'ProstateX-0243_06-04-2011',
    'ProstateX-0244_07-01-2011',
    'ProstateX-0246_07-01-2011',
    'ProstateX-0247_07-06-2011',
    'ProstateX-0248_07-27-2011',
    'ProstateX-0249_08-11-2011',
    'ProstateX-0250_08-17-2011',
    'ProstateX-0251_08-17-2011',
    'ProstateX-0252_08-17-2011',
    'ProstateX-0253_08-17-2011',
    'ProstateX-0254_08-18-2011',
    'ProstateX-0255_08-18-2011',
    'ProstateX-0257_08-18-2011',
    'ProstateX-0258_08-31-2011',
    'ProstateX-0259_08-31-2011',
    'ProstateX-0260_09-08-2011',
    'ProstateX-0261_09-15-2011',
    'ProstateX-0262_09-15-2011',
    'ProstateX-0263_09-22-2011',
    'ProstateX-0264_09-30-2011',
    'ProstateX-0265_10-21-2011',
    'ProstateX-0266_11-15-2011',
    'ProstateX-0268_12-03-2011',
    'ProstateX-0270_12-11-2011',
    'ProstateX-0269_12-03-2011',
    'ProstateX-0271_12-14-2011',
    'ProstateX-0272_12-14-2011',
    'ProstateX-0273_12-14-2011',
    'ProstateX-0274_12-22-2011',
]

# Re-convert only the Bs (diffusion) series without reversing file order
for study in tqdm(reconv_studies):
    series = dicom_series[study]

    study_folder = Path(nifti_root) / study
    study_folder.mkdir(parents=True, exist_ok=True)

    patient = study.split("_")[0]
    study_date = study.split("_")[1]
    patient_dicom_root = Path(dicom_root) / patient

    for folder in patient_dicom_root.iterdir():
        if folder.name[:10] != study_date:
            continue

        for modality, series_name in series.items():
            if modality != 'Bs':
                continue

            dicom_series_folder = folder / series_name

            # Group DICOM files by b-value
            b_value_groups = defaultdict(list)
            dicom_files = list(dicom_series_folder.glob('*.dcm'))

            for dicom_file_path in dicom_files:
                dicom_file = pydicom.dcmread(dicom_file_path)
                b_value_str = dicom_file[(0x0018, 0x0024)].value
                b_value = 'B' + b_value_str.split('_')[1][1:][:-1]
                b_value_groups[b_value].append(str(dicom_file_path))

            # Convert without reversing file paths (original DICOM order)
            for b_value, file_paths in b_value_groups.items():
                reader = sitk.ImageSeriesReader()
                reader.SetFileNames(file_paths)
                image = reader.Execute()

                nifti_filename = study_folder / f'{b_value}.nii.gz'
                sitk.WriteImage(image, str(nifti_filename))

print("Re-conversion completed.")